# Threshold Agent Effect

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import zarr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN = ROOT / "artifacts/simulation_runs/run-REPLACE_WITH_PAIRED_RUN.zarr"
Y = "centered_mean_welfare"

WITHOUT_THRESHOLD_COUNT = 0
WITH_THRESHOLD_COUNT = 1
WELFARE_THRESHOLD = 0.8  # Set to None to use all exact pairs.

In [ ]:
def load_run(path):
    root = zarr.open_group(str(path), mode="r")
    names = root.attrs["parameter_names"]
    params = np.asarray(root["metadata/parameters"], dtype=float)
    input_index = np.asarray(root["results/input_index"], dtype=int)
    n_agents = np.asarray(root["metadata/n_agents"], dtype=int)
    _, n_compositions, n_rows, _ = params.shape

    n_idx = input_index // (n_compositions * n_rows)
    composition_idx = input_index % (n_compositions * n_rows) // n_rows
    row_idx = input_index % n_rows
    df = pd.DataFrame(params[n_idx, composition_idx, row_idx], columns=names)
    df["n_agents"] = n_agents[n_idx]
    df["n_agents_index"] = n_idx
    df["composition_index"] = composition_idx
    df["parameter_row"] = row_idx
    df[Y] = np.asarray(root[f"results/outputs/{Y}.mean"][:, -1], dtype=float)
    return df


run = load_run(RUN).rename(
    columns={
        "threshold_link.link_threshold": "link_threshold",
        "best_response.outside_optimism": "outside_optimism",
        "best_response.exploration": "exploration",
    }
)
run["log_exploration"] = np.log10(run["exploration"])

In [ ]:
PAIR_KEYS = ["n_agents_index", "parameter_row"]

without_threshold = run.loc[
    run["threshold_link_count"] == WITHOUT_THRESHOLD_COUNT,
    PAIR_KEYS + [Y],
].rename(columns={Y: "welfare_without_threshold"})

with_threshold = (
    run.loc[run["threshold_link_count"] == WITH_THRESHOLD_COUNT]
    .copy()
    .rename(columns={Y: "welfare_with_threshold"})
)

paired = with_threshold.merge(
    without_threshold,
    on=PAIR_KEYS,
    how="inner",
    validate="one_to_one",
)
paired["d_welfare"] = (
    paired["welfare_with_threshold"] - paired["welfare_without_threshold"]
)

if WELFARE_THRESHOLD is None:
    analysis = paired.copy()
else:
    analysis = paired.loc[
        paired["welfare_with_threshold"] > WELFARE_THRESHOLD
    ].copy()

if analysis.empty:
    raise ValueError("No paired rows remain after applying WELFARE_THRESHOLD")

analysis["link_centered"] = (
    analysis["link_threshold"] - analysis["link_threshold"].mean()
)

agent_values = [int(n) for n in sorted(analysis.n_agents.unique())]
for n in agent_values:
    analysis[f"effect_n{n}"] = (analysis.n_agents == n).astype(int)
    analysis[f"threshold_slope_n{n}"] = (
        analysis[f"effect_n{n}"] * analysis.link_centered
    )

print(f"{WELFARE_THRESHOLD=}")
print(f"paired rows: {len(paired):,}; analysis rows: {len(analysis):,}")

In [ ]:
analysis

In [ ]:
effect_terms = [f"effect_n{n}" for n in agent_values]
slope_terms = [f"threshold_slope_n{n}" for n in agent_values]
terms = effect_terms + slope_terms

fit = smf.ols(
    "d_welfare ~ " + " + ".join(terms) + " - 1",
    data=analysis,
).fit(cov_type="HC1")

print("effect_n*: threshold-agent effect at average link_threshold")
print("threshold_slope_n*: extra effect of increasing link_threshold by 1")

fit.summary()